In [2]:
!uv run python --version

Python 3.12.11


In [18]:
# インストールする
!uv add python-dotenv json_repair rouge numpy pandas tiktoken nltk pipmaster

Resolved 175 packages in 775ms
Prepared 3 packages in 1.48s
Uninstalled 1 package in 1ms
Installed 3 packages in 10ms
 + ascii-colors==0.11.4
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + pipmaster==1.0.9


In [16]:
!uv add sentence_transformers

Resolved 173 packages in 1.03s
Prepared 13 packages in 25.02s
Uninstalled 1 package in 2ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 16 packages in 8.43s
 + filelock==3.20.0
 + fsspec==2025.9.0
 + huggingface-hub==0.36.0
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + mpmath==1.3.0
 + networkx==3.5
 + pillow==12.0.0
 + safetensors==0.6.2
 + scikit-learn==1.7.2
 + scipy==1.16.2
 + sentence-transformers==5.1.2
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 + tokenizers==0.22.1
 + torch==2.9.0
 + transformers==4.57.1


In [22]:
!uv add openai tenacity

Resolved 183 packages in 159ms
Prepared 1 package in 1.22s
Uninstalled 1 package in 1ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 37ms
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + tenacity==9.1.2


In [10]:
# feature-metadata-filtering ブランチでOK
!git branch

* feature-metadata-filtering
  main
  metadata-filter


In [24]:
# 必要なライブラリのインポート
import os
import tempfile
from minirag import MiniRAG, QueryParam
from minirag.llm.hf import (
    hf_model_complete,
    hf_embed,
)
# from minirag.llm.openai import openrouter_openai_complete
from minirag.llm.openai import openai_complete_if_cache
from minirag.utils import EmbeddingFunc
from minirag.utils import (
    wrap_embedding_func_with_attrs,
    locate_json_string_body_from_string,
    safe_unicode_decode,
    logger,
)
from transformers import AutoModel, AutoTokenizer
import asyncio
import warnings
warnings.filterwarnings('ignore')

## 環境変数

In [29]:
import os
from dotenv import load_dotenv

# .envファイルを読み込む
load_dotenv()

# 環境変数を取得する
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')


os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY


print(f"GEMINI_API_KEY: {GEMINI_API_KEY[:5]}********************")
print(f"HF_TOKEN: {HF_TOKEN[:5]}********************")
print(f"OPENROUTER_API_KEY: {OPENROUTER_API_KEY[:5]}********************")

GEMINI_API_KEY: AIzaS********************
HF_TOKEN: hf_UK********************
OPENROUTER_API_KEY: sk-or********************


## セッティング

In [31]:
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
device = "cuda" if torch.cuda.is_available() else "cpu"
# model = SentenceTransformer("cl-nagoya/ruri-v3-30m", device=device)

# Ruri v3 employs a 1+3 prefix scheme to distinguish between different types of text inputs:
# "" (empty string) is used for encoding semantic meaning.
# "トピック: " is used for classification, clustering, and encoding topical information.
# "検索クエリ: " is used for queries in retrieval tasks.
# "検索文書: " is used for documents to be retrieved.
sentences = [
    "川べりでサーフボードを持った人たちがいます",
    "サーファーたちが川べりに立っています",
    "トピック: 瑠璃色のサーファー",
    "検索クエリ: 瑠璃色はどんな色？",
    "検索文書: 瑠璃色（るりいろ）は、紫みを帯びた濃い青。名は、半貴石の瑠璃（ラピスラズリ、英: lapis lazuli）による。JIS慣用色名では「こい紫みの青」（略号 dp-pB）と定義している[1][2]。",
]

embeddings = model.encode(sentences, convert_to_tensor=True)
print(embeddings.size())
# [5, 256]

similarities = F.cosine_similarity(embeddings.unsqueeze(0), embeddings.unsqueeze(1), dim=2)
print(similarities)
# [[1.0000, 0.9540, 0.8512, 0.7322, 0.7274],
#  [0.9540, 1.0000, 0.8531, 0.7437, 0.7305],
#  [0.8512, 0.8531, 1.0000, 0.8910, 0.8649],
#  [0.7322, 0.7437, 0.8910, 1.0000, 0.9479],
#  [0.7274, 0.7305, 0.8649, 0.9479, 1.0000]]


NameError: name 'model' is not defined

In [32]:
device

'cpu'